# 第七课：配置、环境变量与日志

本课目标：

- 区分代码与运行配置；
- 使用`os.getenv()`读取环境变量；
- 使用`.env`保存本地开发配置；
- 避免把API密钥提交到Git；
- 使用`logging`记录程序运行信息；
- 理解日志级别和密钥脱敏。

这些能力会直接用于后续FastAPI和大模型API项目。

## 1. 为什么配置不应写死在代码中

下面的写法存在安全和维护问题：

```python
API_KEY = "真实密钥"
DATABASE_URL = "生产数据库地址"
```

代码会进入Git历史，而配置会因电脑、环境和部署方式不同而变化。正确思路是让代码从外部读取配置。

## 2. 环境变量

Python通过`os.environ`和`os.getenv()`读取当前进程的环境变量：

In [ ]:
import os

os.environ["DEMO_APP_NAME"] = "my-ai-app"

print(os.getenv("DEMO_APP_NAME"))
print(os.getenv("MISSING_VALUE"))
print(os.getenv("MISSING_VALUE", "default-value"))

`os.getenv("NAME")`找不到变量时返回`None`；第二个参数可以提供默认值：

```python
app_name = os.getenv("APP_NAME", "ai-learning-app")
```

环境变量读取出来通常是字符串。端口、布尔值等配置需要自行转换和验证。

## 3. `.env`与`.env.example`

本地开发时，可以把配置写入`.env`：

```text
APP_NAME=ai-learning-app
LOG_LEVEL=INFO
LLM_API_KEY=真实密钥
```

`python-dotenv`的`load_dotenv()`可以把它加载到环境变量。

- `.env`包含真实值，不能提交；
- `.env.example`只包含变量名和安全示例，需要提交；
- 新成员复制`.env.example`为`.env`并填写自己的值。

仓库根目录的`.gitignore`已经忽略`.env`和`.env.*`，同时允许`.env.example`。

## 4. 为什么使用`logging`

`print()`只负责输出内容；日志系统还能提供级别、时间、模块和输出目标等信息：

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

logger = logging.getLogger("lesson07")
logger.debug("调试细节")
logger.info("应用正常启动")
logger.warning("配置即将过期")
logger.error("请求失败")

## 5. 日志级别

从低到高：

```text
DEBUG → INFO → WARNING → ERROR → CRITICAL
```

日志级别设置为`INFO`时，`DEBUG`不会输出，`INFO`及以上会输出。

常见使用：

- `DEBUG`：开发时的详细变量和流程；
- `INFO`：正常启动、任务完成等重要事件；
- `WARNING`：程序还能继续，但出现值得关注的情况；
- `ERROR`：某次操作失败；
- `CRITICAL`：系统可能无法继续运行。

日志推荐使用参数化写法：

```python
logger.info("用户登录：%s", user_id)
```

而不是提前拼接字符串。更重要的是：不要记录完整API密钥、密码、访问令牌或用户敏感数据。

## 6. 动手作业

1. 在本课目录安装依赖：

```powershell
python -m pip install -r requirements.txt
```

2. 复制`.env.example`并命名为`.env`；
3. 在`.env`中把`LLM_API_KEY`换成练习值，例如`abcd12345678wxyz`，不要填写真实密钥；
4. 完成`app_settings.py`中的三个TODO；
5. 运行`python app_settings.py`，日志中只能出现脱敏密钥。

In [ ]:
import importlib
import os

import app_settings

importlib.reload(app_settings)

os.environ["APP_NAME"] = "test-app"
os.environ["LOG_LEVEL"] = "warning"
os.environ["LLM_API_KEY"] = "abcd12345678wxyz"

settings = app_settings.load_settings()
assert settings.app_name == "test-app"
assert settings.log_level == "WARNING"
assert settings.api_key == "abcd12345678wxyz"
assert settings.masked_api_key() == "abcd...wxyz"

print("配置测试通过")

In [ ]:
os.environ["LOG_LEVEL"] = "UNKNOWN"
try:
    app_settings.load_settings()
except ValueError:
    pass
else:
    raise AssertionError("非法LOG_LEVEL应该抛出ValueError")

os.environ["LOG_LEVEL"] = "INFO"
os.environ["LLM_API_KEY"] = "   "
try:
    app_settings.load_settings()
except RuntimeError:
    pass
else:
    raise AssertionError("空API密钥应该抛出RuntimeError")

print("异常测试通过")

## 7. 理解题

1. 为什么API密钥不应该直接写在Python代码中？
2. `.env`和`.env.example`分别有什么作用，哪一个应该上传GitHub？
3. 当日志级别设置为`WARNING`时，哪些级别会输出？
4. 为什么日志中不能直接记录完整API密钥？